<a href="https://colab.research.google.com/github/detektor777/colab_list_image/blob/main/grl.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#@title ##**Install** { display-mode: "form" }
import os, json, urllib.request

%cd /content
if not os.path.isdir('/content/GRL-Image-Restoration'):
    !git clone --depth 1 https://github.com/ofsoundof/GRL-Image-Restoration.git
%cd /content/GRL-Image-Restoration

# Install dependencies for GRL Transformer
!pip install -q timm einops fvcore yacs gdown opencv-python matplotlib tqdm torchvision fairscale omegaconf

import torch
print('torch:', torch.__version__, '| CUDA available:', torch.cuda.is_available())

# --- GRL Weights (JPEG Compression Artifact Removal) ---------------
os.makedirs('model_zoo', exist_ok=True)
weight_path = 'model_zoo/grl_jpeg_car.ckpt'

# Optional: if you have your own Google Drive ID with weights, enter it here
GDRIVE_ID = ""  #@param {type:"string"}

#@markdown ---

color_mode = "color"  #@param ["color", "grayscale"]
quality_factor = "30"  #@param ["10", "20", "30", "40"]

# Models without the qNN suffix ("jpeg_grl_small_c1.ckpt"/"c3.ckpt") are "blind"
# models with a non-standard input (extra channel for quality map), for which
# there is no ready preprocessing in this notebook. Therefore, we download the model
# for a specific quality_factor - it has a regular input (1 or 3 channels).
channel_prefix = 'c1' if color_mode == 'grayscale' else 'c3'
target_asset_name = f'jpeg_grl_small_{channel_prefix}q{quality_factor}.ckpt'
print(f'🎯 Target weights file: {target_asset_name}')

if os.path.exists(weight_path):
    print('⚠️ File already downloaded. If you changed quality_factor, delete the old one: !rm model_zoo/grl_jpeg_car.ckpt')
elif GDRIVE_ID:
    import gdown
    print('Downloading GRL weights from Google Drive...')
    gdown.download(f'https://drive.google.com/uc?id={GDRIVE_ID}', weight_path, quiet=False)
    print('✅ Weights downloaded.')
else:
    print(f'\n🔎 GDRIVE_ID is not set — looking for {target_asset_name} in GitHub Releases of the repository...')
    api_url = "https://api.github.com/repos/ofsoundof/GRL-Image-Restoration/releases/tags/v1.0.0"
    try:
        req = urllib.request.Request(api_url, headers={"User-Agent": "colab-grl-installer"})
        with urllib.request.urlopen(req) as r:
            release = json.load(r)
        assets = release.get('assets', [])

        print('=== DEBUG: available release files ===')
        for a in assets:
            print(' -', a['name'])

        asset = next((a for a in assets if a['name'] == target_asset_name), None)

        if asset is None:
            candidates = [a for a in assets
                          if a['name'].startswith(f'jpeg_grl_small_{channel_prefix}q')]
            if candidates:
                asset = candidates[0]
                print(f"⚠️ Exact match for quality_factor={quality_factor} not found, "
                      f"choosing the nearest variant: {asset['name']}")

        if asset:
            print(f"\n⬇️ Downloading {asset['name']} ...")
            urllib.request.urlretrieve(asset['browser_download_url'], weight_path)
            print('✅ Weights downloaded to', weight_path)
        else:
            print(f'⚠️ Could not find {target_asset_name} (or analogue) in the release.')
            print('Available release files:')
            for a in assets:
                print(' -', a['name'], '->', a['browser_download_url'])
            print('\nDownload the required file manually and save it as:')
            print(f'/content/GRL-Image-Restoration/{weight_path}')
    except Exception as e:
        print('⚠️ Failed to fetch release list automatically:', e)
        print('Download the weights manually from:')
        print('https://github.com/ofsoundof/GRL-Image-Restoration/releases/tag/v1.0.0')
        print(f'and save the file as: /content/GRL-Image-Restoration/{weight_path}')

print('\n✅ Setup completed! Dependencies installed successfully.')

In [ ]:
#@title ##**Upload Images** { display-mode: "form" }
import os
import shutil
from google.colab import files

upload_folder = './inputs/user_upload'
result_folder = './results/user_upload'

if os.path.isdir(upload_folder):
    shutil.rmtree(upload_folder)
if os.path.isdir(result_folder):
    shutil.rmtree(result_folder)

os.makedirs(upload_folder, exist_ok=True)
os.makedirs(result_folder, exist_ok=True)

# Upload images — you can select multiple files at once
uploaded = files.upload()
for filename in uploaded.keys():
    dst_path = os.path.join(upload_folder, filename)
    print(f'Moved {filename} -> {dst_path}')
    shutil.move(filename, dst_path)

print(f'\n{len(uploaded)} image(s) uploaded to {upload_folder}')

In [ ]:
#@title ##**Run** { display-mode: "form" }
import os, gc, torch, psutil, time

start_time_main = time.time()

# --- Prevent working directory loss after Restart session --------------
if os.path.basename(os.getcwd()) != 'GRL-Image-Restoration':
    if os.path.isdir('/content/GRL-Image-Restoration'):
        os.chdir('/content/GRL-Image-Restoration')
        print(f'⚠️ Working directory was reset. Switched to {os.getcwd()}')
    else:
        raise RuntimeError('Folder /content/GRL-Image-Restoration not found. Please run the "Install" cell.')

# --- Prevent upload_folder/result_folder variables loss ---------------
if 'upload_folder' not in globals():
    upload_folder = './inputs/user_upload'
if 'result_folder' not in globals():
    result_folder = './results/user_upload'

if not os.path.isdir(upload_folder) or not os.listdir(upload_folder):
    raise RuntimeError(f'No images found in {upload_folder}. Please run "Upload Images".')
os.makedirs(result_folder, exist_ok=True)

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
gc.collect()
torch.cuda.empty_cache()
print(f'Free RAM: {psutil.virtual_memory().available/1024**3:.2f} GiB')

# --- Tile settings (fix for CUDA OOM on large images) -------------------

tile_size = 288  #@param {type:"integer"}
tile_overlap = 32  #@param {type:"integer"}

# --- Generate custom inference script ---
inference_script = """
import os
import sys
import argparse
import traceback
import torch
import numpy as np
from PIL import Image
from collections import OrderedDict
from tqdm import tqdm

sys.path.append('.')

# --- Correct import path for the GRL architecture --------------------------
# Confirmed in config/model/grl/grl_small.yaml: _target_: models.networks.grl.GRL
from models.networks.grl import GRL

parser = argparse.ArgumentParser()
parser.add_argument('--input', required=True)
parser.add_argument('--output', required=True)
parser.add_argument('--ckpt', default='model_zoo/grl_jpeg_car.ckpt')
parser.add_argument('--tile', type=int, default=288)
parser.add_argument('--tile_overlap', type=int, default=32)
args = parser.parse_args()

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

# --- Loading checkpoint -------------------------------------------------
print(f'=== DEBUG: loading checkpoint {args.ckpt} ===')
raw_state = torch.load(args.ckpt, map_location='cpu')

# The checkpoint can be either a bare state_dict or wrapped in
# {'state_dict': ...} / {'params': ...} (depending on how it was saved).
if isinstance(raw_state, dict) and 'state_dict' in raw_state:
    raw_state = raw_state['state_dict']
elif isinstance(raw_state, dict) and 'params' in raw_state:
    raw_state = raw_state['params']

# Keys of format "model.conv_first.weight" -> "conv_first.weight"
# (pytorch-lightning GRL module wrapper inside self.model)
state_dict = OrderedDict()
for k, v in raw_state.items():
    new_k = k[len('model.'):] if k.startswith('model.') else k
    state_dict[new_k] = v

print('=== DEBUG: sample keys after cleanup ===')
for i, k in enumerate(state_dict.keys()):
    if i >= 5:
        break
    print(' -', k, tuple(state_dict[k].shape))

# We determine the model's in_channels directly from the conv_first weights to avoid guessing
in_channels = state_dict['conv_first.weight'].shape[1]
print(f'=== DEBUG: detected in_channels from checkpoint = {in_channels} ===')

# --- GRL-small architecture for JPEG Compression Artifact Removal
# (taken from config/experiment/jpeg/grl/grl_p288.yaml + config/model/grl/grl_small.yaml)
model = GRL(
    upscale=1,
    in_channels=in_channels,
    img_size=288,
    img_range=1.0,
    upsampler="",
    embed_dim=128,
    depths=[4, 4, 4, 4],
    num_heads_window=[2, 2, 2, 2],
    num_heads_stripe=[2, 2, 2, 2],
    window_size=36,
    stripe_size=[72, 144],
    stripe_groups=[None, None],
    stripe_shift=True,
    mlp_ratio=2,
    qkv_proj_type="linear",
    anchor_proj_type="avgpool",
    anchor_one_stage=True,
    anchor_window_down_factor=4,
    out_proj_type="linear",
    conv_type="1conv",
    init_method="n",
    local_connection=False,
)

missing, unexpected = model.load_state_dict(state_dict, strict=False)
print(f'=== DEBUG: missing keys: {len(missing)}, unexpected keys: {len(unexpected)} ===')
if missing:
    print('Missing (first 10):', missing[:10])
if unexpected:
    print('Unexpected (first 10):', unexpected[:10])

model.eval().to(device)

# --- Tiled inference without seam blurring (crop-and-stitch) ------------
# The overlap is used only as context for the model; only the clean
# central part of each tile is written to the result — avoiding overlap
# averaging which caused overall blurring of the image.
@torch.no_grad()
def tile_inference(model, img_tensor, tile, tile_overlap):
    b, c, h, w = img_tensor.shape
    tile = min(tile, h, w)
    tile_overlap = tile_overlap - (tile_overlap % 2)  # make it even
    half = tile_overlap // 2
    stride = max(tile - tile_overlap, 1)

    h_idx_list = sorted(set(list(range(0, max(h - tile, 0), stride)) + [max(h - tile, 0)]))
    w_idx_list = sorted(set(list(range(0, max(w - tile, 0), stride)) + [max(w - tile, 0)]))

    out = torch.zeros(b, c, h, w, device=img_tensor.device)

    for i, h_idx in enumerate(h_idx_list):
        core_top = 0 if i == 0 else half
        core_bottom = tile if i == len(h_idx_list) - 1 else tile - half
        abs_top, abs_bottom = h_idx + core_top, h_idx + core_bottom
        for j, w_idx in enumerate(w_idx_list):
            core_left = 0 if j == 0 else half
            core_right = tile if j == len(w_idx_list) - 1 else tile - half
            abs_left, abs_right = w_idx + core_left, w_idx + core_right

            in_patch = img_tensor[..., h_idx:h_idx + tile, w_idx:w_idx + tile]
            out_patch = model(in_patch)

            out[..., abs_top:abs_bottom, abs_left:abs_right] = \
                out_patch[..., core_top:core_bottom, core_left:core_right]

    return out

# --- Inference -------------------------------------------------------
os.makedirs(args.output, exist_ok=True)
image_extensions = {'.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff'}
files_in = sorted([f for f in os.listdir(args.input)
                    if os.path.splitext(f.lower())[1] in image_extensions])

if not files_in:
    print(f'⚠️ No images found in {args.input}')

for fname in tqdm(files_in, desc='Processing'):
    path_in = os.path.join(args.input, fname)
    img = Image.open(path_in)
    img = img.convert('L') if in_channels == 1 else img.convert('RGB')

    arr = np.array(img).astype(np.float32) / 255.0
    if arr.ndim == 2:
        arr = arr[:, :, None]
    tensor = torch.from_numpy(arr).permute(2, 0, 1).unsqueeze(0).to(device)

    with torch.no_grad():
        try:
            out = tile_inference(model, tensor, tile=args.tile, tile_overlap=args.tile_overlap)
        except Exception:
            print(f'❌ Error while running model on {fname}:')
            traceback.print_exc()
            continue

    out = out.clamp(0, 1).squeeze(0).permute(1, 2, 0).cpu().numpy()
    out = (out * 255.0).round().astype(np.uint8)
    if out.shape[2] == 1:
        out = out[:, :, 0]
    Image.fromarray(out).save(os.path.join(args.output, fname))
    print(f'✅ Saved: {os.path.join(args.output, fname)}')

print('=== Inference finished ===')
"""

with open('inference_colab.py', 'w', encoding='utf-8') as f:
    f.write(inference_script)

# Run inference
!python inference_colab.py --input {upload_folder} --output {result_folder} --ckpt model_zoo/grl_jpeg_car.ckpt --tile {tile_size} --tile_overlap {tile_overlap}

# --- Calculate and display execution stats ----------------------------------
elapsed_time = time.time() - start_time_main
m, s = divmod(elapsed_time, 60)
h, m = divmod(m, 60)

time_str = ""
if h > 0:
    time_str += f"{int(h)}h "
if m > 0 or h > 0:
    time_str += f"{int(m)}m "
time_str += f"{s:.1f}s"

print(f'\n✅ Done! Processed images -> {result_folder}')
print(f'⏱️ Total execution time: {time_str}')
print(f'🖥️ RAM available after run: {psutil.virtual_memory().available/1024**3:.2f} GiB')

In [ ]:
#@title ##**Visualization** { display-mode: "form" }
import os
import base64
from io import BytesIO
import PIL.Image
from IPython.display import HTML, display

def is_image_file(filename):
    image_extensions = {'.png', '.jpg', '.jpeg', '.gif', '.bmp', '.tiff'}
    return os.path.splitext(filename.lower())[1] in image_extensions

def resize_image_maintain_aspect(image, max_width, target_height=None):
    width, height = image.size
    if width > max_width:
        new_height = int(height * max_width / width)
        image = image.resize((max_width, new_height), PIL.Image.Resampling.LANCZOS)
    if target_height is not None and image.size[1] != target_height:
        new_width = int(image.size[0] * target_height / image.size[1])
        image = image.resize((new_width, target_height), PIL.Image.Resampling.LANCZOS)
    return image

def image_to_base64(image):
    buffered = BytesIO()
    image.save(buffered, format="PNG")
    return base64.b64encode(buffered.getvalue()).decode('utf-8')

def find_images(root):
    found = []
    for dirpath, _, fnames in os.walk(root):
        for fname in sorted(fnames):
            if is_image_file(fname):
                found.append(os.path.join(dirpath, fname))
    return sorted(found)

filenames_input = sorted([f for f in os.listdir(upload_folder) if is_image_file(f)])
paths_output = find_images(result_folder)

if not filenames_input or not paths_output:
    print(f'Error: no images found in {upload_folder} or {result_folder}.')
else:
    for filename, path_output in zip(filenames_input, paths_output):
        try:
            image_before = PIL.Image.open(os.path.join(upload_folder, filename))
            image_after = PIL.Image.open(path_output)

            max_width = min(image_after.size[0], 1000)
            image_after = resize_image_maintain_aspect(image_after, max_width)
            target_height = image_after.size[1]
            image_before = resize_image_maintain_aspect(image_before, max_width, target_height)

            if image_before.mode != 'RGB':
                image_before = image_before.convert('RGB')
            if image_after.mode != 'RGB':
                image_after = image_after.convert('RGB')

            before_base64 = image_to_base64(image_before)
            after_base64 = image_to_base64(image_after)

            html_code = f"""
            <div style="position: relative; width: {image_after.size[0]}px; height: {image_after.size[1]}px; margin-bottom: 20px;">
                <div style="position: relative; width: 100%; height: 100%; overflow: hidden;">
                    <img src="data:image/png;base64,{before_base64}" style="position: absolute; width: 100%; height: 100%;">
                    <div style="position: absolute; width: 100%; height: 100%; overflow: hidden; clip-path: inset(0 0 0 50%);">
                        <img src="data:image/png;base64,{after_base64}" style="position: absolute; width: 100%; height: 100%;">
                    </div>
                </div>
                <div class="slider" style="position: absolute; top: 0; bottom: 0; width: 4px; background: white; cursor: ew-resize; left: 50%; box-shadow: 0 0 5px rgba(0,0,0,0.5);">
                    <div style="position: absolute; top: 50%; transform: translateY(-50%); width: 20px; height: 20px; background: white; border-radius: 50%; left: -8px;"></div>
                </div>
                <div style="position: absolute; top: 8px; left: 8px; color: white; font-family: sans-serif; font-size: 14px; text-shadow: 0 1px 3px rgba(0,0,0,0.8);">Before</div>
                <div style="position: absolute; top: 8px; right: 8px; color: white; font-family: sans-serif; font-size: 14px; text-shadow: 0 1px 3px rgba(0,0,0,0.8);">After</div>
            </div>
            <script>
                document.querySelectorAll('.slider').forEach(slider => {{
                    let isDragging = false;
                    const container = slider.parentElement.querySelector('div:nth-child(1)');
                    const clipDiv = container.querySelector('div');

                    slider.addEventListener('mousedown', (e) => {{ isDragging = true; e.preventDefault(); }});
                    document.addEventListener('mouseup', () => {{ isDragging = false; }});
                    document.addEventListener('mousemove', (e) => {{
                        if (!isDragging) return;
                        const rect = container.getBoundingClientRect();
                        let x = e.clientX - rect.left;
                        if (x < 0) x = 0;
                        if (x > rect.width) x = rect.width;
                        const percentage = (x / rect.width) * 100;
                        slider.style.left = percentage + '%';
                        clipDiv.style.clipPath = `inset(0 0 0 ${{percentage}}%)`;
                    }});

                    slider.addEventListener('touchstart', (e) => {{ isDragging = true; e.preventDefault(); }});
                    document.addEventListener('touchend', () => {{ isDragging = false; }});
                    document.addEventListener('touchmove', (e) => {{
                        if (!isDragging) return;
                        const rect = container.getBoundingClientRect();
                        let x = e.touches[0].clientX - rect.left;
                        if (x < 0) x = 0;
                        if (x > rect.width) x = rect.width;
                        const percentage = (x / rect.width) * 100;
                        slider.style.left = percentage + '%';
                        clipDiv.style.clipPath = `inset(0 0 0 ${{percentage}}%)`;
                    }});
                }});
            </script>
            """

            display(HTML(html_code))

        except Exception as e:
            print(f'Error processing {filename} and {path_output}: {e}')

In [ ]:
#@title ##**Download Results** { display-mode: "form" }
import os
from google.colab import files

# Fallback to the default folder if the variable was lost due to session restart
if 'result_folder' not in globals():
    result_folder = './results/user_upload'

if not os.path.exists(result_folder):
    print(f"⚠️ Error: Directory '{result_folder}' does not exist.")
else:
    # Filter only files (ignoring subdirectories)
    processed_files = [f for f in os.listdir(result_folder) if os.path.isfile(os.path.join(result_folder, f))]

    if len(processed_files) == 0:
        print("⚠️ No processed images found to download.")
    elif len(processed_files) == 1:
        # Download the single file directly
        single_file_path = os.path.join(result_folder, processed_files[0])
        print(f"📥 Downloading single image directly: {processed_files[0]}")
        files.download(single_file_path)
    else:
        # Package multiple files into a ZIP archive
        zip_filename = 'GRL_result.zip'
        if os.path.exists(zip_filename):
            os.remove(zip_filename)

        print(f"📥 Packaging {len(processed_files)} images into a ZIP archive...")
        os.system(f"zip -r -j {zip_filename} {result_folder}/*")
        files.download(zip_filename)